# 🤖 Notebook 3 — Churn Prediction Model
**Project:** Real-Time Customer Churn Analysis Agent  
**Input:** `data/telco_churn_features.csv`  
**Output:** `models/best_model.pkl` + evaluation plots

### Models trained
| # | Model | Why it's standard for Telco churn |
|---|-------|-----------------------------------|
| 1 | Logistic Regression | Fast, interpretable baseline |
| 2 | Random Forest | Best accuracy/interpretability tradeoff; industry standard |
| 3 | XGBoost | Gradient boosted trees; often top performer on tabular data |
| 4 | **Tuned Random Forest** | Grid-searched best RF — our final model |

---
### Table of Contents
1. [Imports & Load](#1)
2. [Train/Test Split + Class Imbalance](#2)
3. [Baseline — Logistic Regression](#3)
4. [Random Forest](#4)
5. [XGBoost](#5)
6. [Model Comparison](#6)
7. [Hyperparameter Tuning — Best Model](#7)
8. [Final Evaluation & Plots](#8)
9. [Feature Importance — Tuned RF](#9)
10. [Save Final Model](#10)

---
## 1. Imports & Load <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pickle, json, warnings, os

from sklearn.model_selection   import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model      import LogisticRegression
from sklearn.ensemble          import RandomForestClassifier
from sklearn.metrics           import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, ConfusionMatrixDisplay, PrecisionRecallDisplay
)
from sklearn.utils             import class_weight
from imblearn.over_sampling    import SMOTE          # pip install imbalanced-learn

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('⚠  XGBoost not installed — run: pip install xgboost')

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
SEED = 42

# ── load data ──────────────────────────────────────────────────────────────────
df = pd.read_csv('../data/telco_churn_features.csv')

X = df.drop(columns='Churn')
y = df['Churn']

print(f'Dataset  : {X.shape[0]:,} rows × {X.shape[1]} features')
print(f'Churn rate: {y.mean()*100:.1f} %')

---
## 2. Train / Test Split + Handle Class Imbalance <a id='2'></a>

In [ ]:
# stratified split so churn ratio is preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f'Train : {X_train.shape[0]:,} rows  |  churn rate {y_train.mean()*100:.1f}%')
print(f'Test  : {X_test.shape[0]:,} rows  |  churn rate {y_test.mean()*100:.1f}%')

In [ ]:
# ── SMOTE on TRAINING set only (never on test) ─────────────────────────────────
smote = SMOTE(random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'After SMOTE → Train : {X_train_sm.shape[0]:,} rows')
print(f'Class distribution  : {pd.Series(y_train_sm).value_counts().to_dict()}')

# bar chart
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, (ydata, title) in zip(axes, [
    (y_train,    'Before SMOTE (train)'),
    (y_train_sm, 'After  SMOTE (train)')
]):
    counts = pd.Series(ydata).value_counts()
    ax.bar(counts.index.astype(str), counts.values,
           color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.4)
    ax.set_title(title)
    ax.set_xlabel('Churn')
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{int(bar.get_height()):,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 3. Baseline — Logistic Regression <a id='3'></a>

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
lr.fit(X_train_sm, y_train_sm)

y_pred_lr  = lr.predict(X_test)
y_prob_lr  = lr.predict_proba(X_test)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['No Churn', 'Churn']))

---
## 4. Random Forest <a id='4'></a>

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    class_weight='balanced',    # extra guard against imbalance
    random_state=SEED,
    n_jobs=-1
)
rf.fit(X_train_sm, y_train_sm)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['No Churn', 'Churn']))

---
## 5. XGBoost <a id='5'></a>

In [ ]:
if XGBOOST_AVAILABLE:
    # scale_pos_weight handles imbalance internally
    neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
    spw = neg / pos

    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=SEED,
        n_jobs=-1
    )
    xgb.fit(X_train_sm, y_train_sm,
            eval_set=[(X_test, y_test)],
            verbose=False)

    y_pred_xgb = xgb.predict(X_test)
    y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

    print('=== XGBoost ===')
    print(classification_report(y_test, y_pred_xgb, target_names=['No Churn', 'Churn']))
else:
    print('XGBoost skipped — not installed.')

---
## 6. Model Comparison <a id='6'></a>

In [ ]:
def evaluate(name, y_true, y_pred, y_prob):
    return {
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred), 4),
        'Recall'   : round(recall_score(y_true, y_pred), 4),
        'F1'       : round(f1_score(y_true, y_pred), 4),
        'ROC-AUC'  : round(roc_auc_score(y_true, y_prob), 4)
    }

results = [
    evaluate('Logistic Regression', y_test, y_pred_lr, y_prob_lr),
    evaluate('Random Forest',       y_test, y_pred_rf, y_prob_rf),
]
if XGBOOST_AVAILABLE:
    results.append(evaluate('XGBoost', y_test, y_pred_xgb, y_prob_xgb))

results_df = pd.DataFrame(results).set_index('Model')
results_df.style.background_gradient(cmap='YlGn', axis=0)

In [ ]:
# ── metric comparison bar chart ────────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']

fig, ax = plt.subplots(figsize=(11, 5))
results_df[metrics].T.plot(kind='bar', ax=ax, edgecolor='white', width=0.7)
ax.set_ylim(0.5, 1.0)
ax.set_xticklabels(metrics, rotation=0)
ax.set_title('Model Comparison — Test Set Metrics', fontsize=13, fontweight='bold')
ax.legend(title='Model', loc='lower right')
ax.axhline(0.80, color='grey', linestyle='--', linewidth=0.8, label='0.80 reference')
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────────
plt.figure(figsize=(7, 6))

models_roc = [
    ('Logistic Regression', y_prob_lr,  'royalblue'),
    ('Random Forest',       y_prob_rf,  'darkorange'),
]
if XGBOOST_AVAILABLE:
    models_roc.append(('XGBoost', y_prob_xgb, 'green'))

for name, prob, color in models_roc:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name}  (AUC = {auc:.3f})', color=color, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
## 7. Hyperparameter Tuning — Best Model (Random Forest) <a id='7'></a>

We tune Random Forest since it consistently performs well and is widely used in churn literature.  
GridSearchCV with 5-fold stratified CV, optimising **F1** (best metric for imbalanced classification).

In [ ]:
param_grid = {
    'n_estimators' : [100, 200, 300],
    'max_depth'    : [6, 10, None],
    'min_samples_leaf': [2, 5, 10],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

grid_rf = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1),
    param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train_sm, y_train_sm)

print('\nBest parameters:')
print(grid_rf.best_params_)
print(f'Best CV F1 : {grid_rf.best_score_:.4f}')

In [ ]:
best_rf = grid_rf.best_estimator_

y_pred_default = best_rf.predict(X_test)
y_prob_best = best_rf.predict_proba(X_test)[:, 1]

print('=== Tuned Random Forest (Default Threshold 0.5) ===')
print(classification_report(y_test, y_pred_default, target_names=['No Churn', 'Churn']))

---
## 8. Final Evaluation & Plots <a id='8'></a>

### 8.1 Threshold analysis — choose the best operating point

> In churn, **recall is often more important than precision** — missing a churner costs more than a false alarm.

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.05)

thresh_results = []
for t in thresholds:
    y_pred_t = (y_prob_best >= t).astype(int)
    thresh_results.append({
        'threshold' : round(t, 2),
        'precision' : precision_score(y_test, y_pred_t, zero_division=0),
        'recall'    : recall_score(y_test, y_pred_t, zero_division=0),
        'f1'        : f1_score(y_test, y_pred_t, zero_division=0),
    })

thresh_df = pd.DataFrame(thresh_results)

plt.figure(figsize=(10, 4))
for metric, color in [('precision','steelblue'), ('recall','#e74c3c'), ('f1','darkorange')]:
    plt.plot(thresh_df['threshold'], thresh_df[metric], marker='o', markersize=4,
             label=metric, color=color, linewidth=2)

best_thresh_row = thresh_df.loc[thresh_df['f1'].idxmax()]
BEST_THRESHOLD = best_thresh_row['threshold']

plt.axvline(BEST_THRESHOLD, color='grey', linestyle='--', label=f'Best F1 threshold = {BEST_THRESHOLD}')

plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title('Precision / Recall / F1 vs Threshold', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print(f'\nOptimal threshold for F1: {BEST_THRESHOLD}')
print(thresh_df[thresh_df['threshold'] == BEST_THRESHOLD].to_string(index=False))

# Apply the best threshold to get final predictions
y_pred_best = (y_prob_best >= BEST_THRESHOLD).astype(int)

### 8.2 Final Evaluation (with Optimal Threshold)

In [ ]:
print(f'=== Tuned Random Forest (Optimal Threshold = {BEST_THRESHOLD}) ===')
print(classification_report(y_test, y_pred_best, target_names=['No Churn', 'Churn']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# raw counts
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=['No Churn', 'Churn'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title(f'Confusion Matrix (counts)\nThreshold: {BEST_THRESHOLD}', fontweight='bold')

# normalised
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=['No Churn', 'Churn'],
    cmap='Oranges', normalize='true', ax=axes[1]
)
axes[1].set_title(f'Confusion Matrix (normalised)\nThreshold: {BEST_THRESHOLD}', fontweight='bold')

plt.suptitle('Tuned Random Forest with Optimal Threshold', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 8.3 ROC + Precision-Recall curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
fpr, tpr, thresholds_roc = roc_curve(y_test, y_prob_best)
auc_val = roc_auc_score(y_test, y_prob_best)
axes[0].plot(fpr, tpr, color='#e74c3c', linewidth=2.5,
             label=f'Tuned RF  AUC = {auc_val:.3f}')
axes[0].fill_between(fpr, tpr, alpha=0.08, color='#e74c3c')
axes[0].plot([0,1],[0,1],'k--', linewidth=0.8)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend()

# Precision-Recall
PrecisionRecallDisplay.from_predictions(y_test, y_prob_best, ax=axes[1],
                                         name='Tuned RF', color='steelblue')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')

plt.suptitle('Final Model — Tuned Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 8.4 Score distribution by class

In [ ]:
prob_df = pd.DataFrame({'prob': y_prob_best, 'Churn': y_test.values})

plt.figure(figsize=(8, 4))
for label, color, name in [(0, '#2ecc71', 'No Churn'), (1, '#e74c3c', 'Churn')]:
    subset = prob_df[prob_df['Churn'] == label]['prob']
    plt.hist(subset, bins=30, alpha=0.6, color=color, label=name, edgecolor='white')

plt.axvline(BEST_THRESHOLD, color='black', linestyle='--',
            label=f'Threshold = {BEST_THRESHOLD}')
plt.xlabel('Predicted Churn Probability')
plt.ylabel('Count')
plt.title('Predicted Probability Distribution by Actual Class', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
## 9. Feature Importance — Tuned RF <a id='9'></a>

In [ ]:
imp_df = pd.DataFrame({
    'feature'   : X.columns,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

top20 = imp_df.head(20)

# colour: engineered features in a different shade
engineered = {'num_services', 'avg_monthly_spend', 'charges_per_month_ratio',
              'risk_score', 'tenure_group', 'has_streaming', 'has_support',
              'is_month_to_month', 'auto_payment'}
colors = ['#e67e22' if f in engineered else 'steelblue' for f in top20['feature']]

plt.figure(figsize=(10, 7))
plt.barh(top20['feature'][::-1], top20['importance'][::-1],
         color=colors[::-1], edgecolor='grey')
plt.xlabel('Mean Decrease Impurity')
plt.title('Top-20 Feature Importances — Tuned RF', fontsize=13, fontweight='bold')

# legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', label='Original feature'),
    Patch(facecolor='#e67e22',   label='Engineered feature')
]
plt.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

print('\nFull importance table:')
print(imp_df.to_string(index=False))

---
## 10. Save Final Model <a id='10'></a>

In [ ]:
os.makedirs('../models', exist_ok=True)

# ── model ─────────────────────────────────────────────────────────────────────
with open('../models/best_model.pkl', 'wb') as f:
    pickle.dump(best_rf, f)
print('✅  Saved → models/best_model.pkl')

# ── model metadata ─────────────────────────────────────────────────────────────
# collect final metrics
model_meta = {
    'model_type'    : 'RandomForestClassifier',
    'best_params'   : grid_rf.best_params_,
    'best_threshold': float(BEST_THRESHOLD),
    'test_metrics'  : {
        'accuracy' : round(accuracy_score(y_test, y_pred_best), 4),
        'precision': round(precision_score(y_test, y_pred_best), 4),
        'recall'   : round(recall_score(y_test, y_pred_best), 4),
        'f1'       : round(f1_score(y_test, y_pred_best), 4),
        'roc_auc'  : round(roc_auc_score(y_test, y_prob_best), 4)
    },
    'feature_columns': X.columns.tolist(),
    'top_5_features' : imp_df.head(5)['feature'].tolist()
}

with open('../models/model_metadata.json', 'w') as f:
    json.dump(model_meta, f, indent=2)
print('✅  Saved → models/model_metadata.json')

print('\n=== Final Model Summary ===')
for k, v in model_meta['test_metrics'].items():
    print(f'  {k:12s}: {v}')
print(f'  threshold   : {BEST_THRESHOLD}')
print(f'  top features: {model_meta["top_5_features"]}')

---
## ✅ Summary

| | Logistic Reg | Random Forest | XGBoost | **Tuned RF** |
|--|--|--|--|--|
| Purpose | Baseline | Strong default | Gradient boost | **Final model** |
| Imbalance handling | SMOTE | SMOTE + balanced | SMOTE + spw | SMOTE + balanced |
| Tuning | None | Default params | Default params | GridSearchCV (F1) |
| Output | — | — | — | `models/best_model.pkl` |

**Key takeaways:**
- Tenure, MonthlyCharges and contract type are the dominant drivers — consistent with churn literature.
- Engineered features (`risk_score`, `avg_monthly_spend`) contributed meaningfully.
- Threshold tuning lets you trade precision ↔ recall based on business cost of false negatives. Applying the optimal threshold directly improves recall, capturing those missing churners.

**Next:** `04_realtime_alert_simulation.ipynb` — stream synthetic events through this model and fire alerts when `risk_score > threshold`.